Benchmark
============

Adding all planners
===========


In [ ]:
import matplotlib.pylab as plt
%matplotlib inline
from IPPerfMonitor import IPPerfMonitor
#import IPBasicPRM
#import IPVISBasicPRM

#import IPVisibilityPRM
#import IPVISVisibilityPRM

#import IPLazyPRM
#import IPVISLazyPRM

#import IPRRT
#import IPVISRRT

import IPAStar
import IPVISAStar

Set-up of the test scenario and the configuration for all planner
===================================

Following a procedure to compare all discussed planners are shown:

1. Configuration for every planner is defined
2. The configuration and the planner is stored in the variable setup, a Python dict()
3. the variable setup is then used to uniformly call the planning calls


In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML, display
import copy
import numpy as np
import matplotlib.pyplot as plt

# Hilfsfunktion (unverändert, aber wir behandeln den Rückgabewert jetzt besser)
def _interpolate_line(startPos, endPos, step_l):
    steps = []
    line = np.array(endPos) - np.array(startPos)
    line_l = np.linalg.norm(line)
    if line_l == 0:
        return [np.array(startPos)]
    step = line / line_l * step_l
    n_steps = int(np.floor(line_l / step_l))
    c_step = np.array(startPos, dtype=float)
    for i in range(n_steps):
        steps.append(copy.deepcopy(c_step))
        c_step = c_step + step
    if not np.allclose(c_step, np.array(endPos)):
        steps.append(np.array(endPos, dtype=float))
    return steps

def animate_solution_multi_downsampled(prm, environment, solution, visualizer=None, 
                                     workSpaceLimits=None, step_l=0.1, interval=50, 
                                     max_frames=250, figsize=(6,6), dpi=70, 
                                     pause_frames=40):
    
    # 1. Basic Checks
    if not solution or len(solution) < 2:
        raise ValueError('solution must contain at least two nodes')

    # 1. Récupération des positions pour l'interpolation du robot
    path_pos = [prm.graph.nodes[n]['pos'] for n in solution]
    
    # Interpolation (Mouvement fluide du point rouge)
    full_path = [path_pos[0]]
    for i in range(1, len(path_pos)):
        s, e = np.array(path_pos[i-1]), np.array(path_pos[i])
        dist = np.linalg.norm(e - s)
        if dist > 0:
            n_steps = int(np.floor(dist / step_l))
            for j in range(1, n_steps + 1):
                full_path.append(s + (e - s) * (j / (n_steps + 1)))
    full_path.append(path_pos[-1])

    # Downsampling
    if len(full_path) > max_frames:
        indices = np.linspace(0, len(full_path)-1, max_frames).astype(int)
        frames_data = [full_path[i] for i in indices]
    else:
        frames_data = full_path

    # 2. Setup Figure
    fig_local, ax = plt.subplots(figsize=figsize, dpi=dpi)
    limits = environment.getEnvironmentLimits()

    def _animate(current_pos):
        ax.cla() # On efface pour redessiner proprement
        
        # --- APPEL DU VISUALISEUR OFFICIEL ---
        # C'est ici que les points bleus et le chemin vert sont dessinés
        # exactement comme dans vos figures statiques.
        visualizer(prm, solution, ax=ax, nodeSize=100)
        
        # On redessine les obstacles par sécurité
        environment.drawObstacles(ax)
        
        # On remet les limites (parce que visualizer peut les changer)
        ax.set_xlim(limits[0])
        ax.set_ylim(limits[1])
        ax.set_title("A* Official Visualization & Animation")

        # --- AJOUT DU ROBOT ROUGE ---
        ax.plot(current_pos[0], current_pos[1], 'o', color='red', 
                markersize=12, markeredgecolor='black', zorder=10)

    print(f"Rendu de l'animation avec le visualiseur officiel...")
    ani = animation.FuncAnimation(fig_local, _animate, frames=frames_data, interval=interval)
    
    html = HTML(ani.to_jshtml())
    display(html)
    plt.close()
    return ani

In [ ]:
plannerFactory = dict()

#basicConfig = dict()
#basicConfig["radius"] = 3
#basicConfig["numNodes"] = 200
#plannerFactory["basePRM"] = [IPBasicPRM.BasicPRM, basicConfig, IPVISBasicPRM.basicPRMVisualize]

#visbilityConfig = dict()
#visbilityConfig["ntry"] = 300
#plannerFactory["visibilityPRM"] = [IPVisibilityPRM.VisPRM, visbilityConfig, IPVISVisibilityPRM.visibilityPRMVisualize ]

#lazyConfig = dict()
#lazyConfig["initialRoadmapSize"] = 10
#lazyConfig["updateRoadmapSize"]  = 5 
#lazyConfig["kNearest"] = 8
#lazyConfig["maxIterations"] = 20
#plannerFactory["lazyPRM"] = [IPLazyPRM.LazyPRM, lazyConfig, IPVISLazyPRM.lazyPRMVisualize]

astarConfig = dict()
astarConfig["heuristic"] = 'euclidean' 
astarConfig["w"]  = 0.7
plannerFactory["astar"] = [IPAStar.AStar, astarConfig, IPVISAStar.aStarVisualizeWspace]

#astarConfig2 = dict()
#astarConfig2["heuristic"] = 'euclidean' 
#astarConfig2["w"]  = 0.9
#plannerFactory["astar2"] = [IPAStar.AStar, astarConfig2, IPVISAStar.aStarVisualize]





In [ ]:
class ResultCollection (object):
    
    def __init__(self, plannerFactoryName, planner, benchmark, solution, perfDataFrame):
        self.plannerFactoryName = plannerFactoryName
        self.planner = planner
        self.benchmark = benchmark
        self.solution = solution
        self.perfDataFrame = perfDataFrame

In [ ]:
import IPTestSuite2 as ts
from shapely.geometry import Point, Polygon, LineString
from shapely import plotting


In [ ]:
#import importlib
#importlib.reload(IPTestSuiteSS2024)

In [ ]:
fullBenchList = ts.benchList

for benchmark in fullBenchList:
    print(benchmark.name)

In [ ]:
for benchmark in fullBenchList:
    fig_local = plt.figure(figsize=(7,7))
    ax = fig_local.add_subplot(1,1,1)
    title = benchmark.name
    ax.set_title(title)
    ax.set_xlim(benchmark.collisionChecker.getEnvironmentLimits()[0])
    ax.set_ylim(benchmark.collisionChecker.getEnvironmentLimits()[1])
    plotting.plot_points(Point(benchmark.startList[0]).buffer(.3), color="g", ax=ax)
    plotting.plot_points(Point(benchmark.goalList[0]).buffer(.3), color="b", ax=ax)
    #try:
    benchmark.collisionChecker.drawObstacles(ax)
  
        
    #except Exception as e:
    #    print ("Error", e)
    #    pass
    plt.show()

In [ ]:
resultList = list()
testList = fullBenchList

for key,producer in list(plannerFactory.items()):
    print(key, producer)
    for benchmark in testList:
        print ("Planning: " + key + " - " + benchmark.name)
        #planner = IPBasicPRM.BasicPRM(benchmark.collisionChecker)
        planner = producer[0](benchmark.collisionChecker)
        IPPerfMonitor.clearData()
        try:
            
            resultList.append(ResultCollection(key,
                                          planner, 
                                           benchmark, 
                                           planner.planPath(benchmark.startList,benchmark.goalList,producer[1]),
                                           IPPerfMonitor.dataFrame()
                                          ),
                        )
        except Exception as e:
        #    throw e
            print ("PLANNING ERROR ! PLANNING ERROR ! PLANNING ERROR ", e)
            pass



In [ ]:
import matplotlib.pyplot as plt

for result in resultList:
    
    fig_local = plt.figure(figsize=(20,20))
    ax = fig_local.add_subplot(1,1,1)
    title = result.plannerFactoryName + " - " + result.benchmark.name
    if result.solution == []:
        title += " (No path found!)"
    title += "\n Assumed complexity level " + str(result.benchmark.level)
    ax.set_title(title)
    try:
        #IPVISBasicsPRM.basicPRMVisualize(result.planner, result.solution, ax=ax, nodeSize=100))
        plannerFactory[result.plannerFactoryName][2](result.planner, result.solution, ax=ax, nodeSize=100)
    except Exception as e:
        print ("Error", e)
        pass
    
    plt.show()

In [ ]:
for bench in testList:
    for result in resultList:
        if result.benchmark.name == bench.name:
            if result.solution is not None :
                print(result.plannerFactoryName)
                animate_solution_multi_downsampled(
                    result.planner, bench.collisionChecker, result.solution,
                    visualizer=plannerFactory[result.plannerFactoryName][2],
                    step_l=0.4, interval=60, max_frames=80, figsize=(20,20), dpi=50)

In [ ]:
import numpy as np
for bench in testList:
    title = bench.name
    pathLength = dict()
    planningTime = dict()
    roadmapSize  = dict()
    
    try:
        for result in resultList:
            if result.benchmark.name == bench.name:
                #print result.benchmark.name  + " - " +  result.plannerFactoryName, len(result.solution)
                pathLength[result.plannerFactoryName] = len(result.solution)
                planningTime[result.plannerFactoryName] = result.perfDataFrame.groupby(["name"]).sum(numeric_only=True)["time"]["planPath"]
                roadmapSize[result.plannerFactoryName] = result.planner.graph.size()


        fig, ax = plt.subplots()

        width = 0.2

        ax.bar(np.arange(len(pathLength.keys())), pathLength.values(),width, color="blue")
        ax.set_ylabel(title + " Number of nodes in path", color="blue")
        ax.set_xticks(np.arange(len(pathLength.keys())) + width)
        ax.set_xticklabels(pathLength.keys())

        ax2 = ax.twinx()
        bar = ax2.bar(np.arange(len(pathLength.keys()))+width, planningTime.values(),width, color="red")
        ax2.set_ylabel(title + " Planning time", color="y")

        # Add coloring and patterns on axis two
        hatches = ['x' if length==0 else '' for length in pathLength.values()]
        color   = ['red' if length==0 else 'yellow' for length in pathLength.values()]
        for i,thisbar in enumerate(bar.patches):
            thisbar.set_facecolor(color[i])
            thisbar.set_hatch(hatches[i])

        # Multiple axes 
        ax3 = ax.twinx()
        ax3.bar(np.arange(len(pathLength.keys()))+2*width, roadmapSize.values(),width, color="purple")
        ax3.set_ylabel(title + " Roadmap size",  color="purple")
        ax3.spines['right'].set_position(('axes', 1.15))
        ax3.spines['right'].set_color("purple")
    except:
        pass

    plt.show()


    
        
    